# Evaluation (PyTorch)

Copyright 2025 Universitat Politècnica de Catalunya

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

**PyTorch copy of `evaluation.ipynb`.** Same cells, same models, same output tables. Every original
TensorFlow line is kept as a `#TF:` comment with its PyTorch translation directly below; lines without
TensorFlow are unchanged. It reads the TF-free datasets in `data_torch/` and the converted checkpoints
`ckpt/paper_weights/**/<checkpoint>.pt` (produced by `convert_tf_checkpoint.py --all-known`). See
`PYTORCH_PORT.md` for the translation notes and `PYTORCH_PARITY.md` for the measured agreement with the
TensorFlow evaluation.

In [ ]:
import os
#TF: os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

#TF: import tensorflow as tf
import torch
import numpy as np
from utils import prepare_targets_and_mask, load_dataset, seg_to_global_reshape
from models import RouteNetGauss
from torch_ragged import sample_to_device

import pickle

# PyTorch-only settings (see PYTORCH_PORT.md, section 5.6)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(max(1, (os.cpu_count() or 2) // 2))
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False


## Load datasets

In [ ]:
#TF: def load_and_format_dataset(path:str, metric:str) -> tf.data.Dataset:
def load_and_format_dataset(path:str, metric:str):
    """Loads and formats the dataset for evaluation

    Parameters
    ----------
    path : str
        Name of the dataset and partition [training/validation/test] to load, in format
        '{name}/{partition}'.
    metric : str
        Metric to load [delay, jitter].

    Returns
    -------
    utils.ListDataset (tf.data.Dataset in the TF version)
        Formatted dataset.
    """
    targets = [
        f"flow_avg_{metric}",
        f"flow_p50_{metric}",
        f"flow_p90_{metric}",
        f"flow_p95_{metric}",
        f"flow_p99_{metric}",
    ]
    mask = f"flow_has_{metric}"
    return load_dataset(path).map(prepare_targets_and_mask(targets, mask))

#TF: def ds_to_dict(ds:tf.data.Dataset) -> dict:
def ds_to_dict(ds) -> dict:
    """Generates a dict with the "sample_idx" feature as key and samples as values.

    Parameters
    ----------
    ds : utils.ListDataset (tf.data.Dataset in the TF version)
        Dataset to convert.

    Returns
    -------
    dict
        Resulting dictionary.
    """
    #TF: return {x["sample_idx"].numpy(): (x, y) for x, y in ds}
    return {int(x["sample_idx"]): (x, y) for x, y in ds}


In [ ]:
ds_mawi_pcaps_delay = load_and_format_dataset("mawi_pcaps/test", "delay")
ds_mawi_pcaps_jitter = load_and_format_dataset("mawi_pcaps/test", "jitter")
ds_mawi_pcaps_simulated_delay = load_and_format_dataset("mawi_pcaps_simulated/test", "delay")
ds_mawi_pcaps_simulated_jitter = load_and_format_dataset("mawi_pcaps_simulated/test", "jitter")

# ds_mawi_pcaps_delay = load_and_format_dataset("mawi_pcaps_new/test", "delay")
# ds_mawi_pcaps_jitter = load_and_format_dataset("mawi_pcaps_new/test", "jitter")
# ds_mawi_pcaps_simulated_delay = load_and_format_dataset("mawi_pcaps_new_simulated/test", "delay")
# ds_mawi_pcaps_simulated_jitter = load_and_format_dataset("mawi_pcaps_new_simulated/test", "jitter")

ds_trex_multiburst_delay = load_and_format_dataset("trex_multiburst/test", "delay")
ds_trex_multiburst_jitter = load_and_format_dataset("trex_multiburst/test", "jitter")
ds_trex_multiburst_simulated_delay = load_and_format_dataset("trex_multiburst_simulated/test", "delay")
ds_trex_multiburst_simulated_jitter = load_and_format_dataset("trex_multiburst_simulated/test", "jitter")

ds_trex_synthetic_delay = load_and_format_dataset("trex_synthetic/test", "delay")
ds_trex_synthetic_jitter = load_and_format_dataset("trex_synthetic/test", "jitter")
ds_trex_synthetic_simulated_delay = load_and_format_dataset("trex_synthetic_simulated/test", "delay")
ds_trex_synthetic_simulated_jitter = load_and_format_dataset("trex_synthetic_simulated/test", "jitter")

## Load models

In [ ]:
def load_model(
    model_id: str,
    checkpoint: str,
    metric: str,
    inference_mode:bool=True,
) -> RouteNetGauss:
    """Loads a SPTGNN_node_queue_v4_multiple_out model from a checkpoint.

    Parameters
    ----------
    model_id : str
        Experiment identifier.
    checkpoint : str
        Checkpoint to load. To be loaded with tf.keras.Model.load_weights.
        PyTorch: the converted state_dict ckpt/<model_id>/<checkpoint>.pt is loaded instead.
    metric : str
        Metric to load [delay, jitter].
    inference_mode : bool, optional
        Activate inference_mode in SPTGNN_node_queue_v4_multiple_out, by default True.

    Returns
    -------
    SPTGNN_node_queue_v4_multiple_out
        Loaded model.
    """
    #TF: optimizer = tf.keras.optimizers.Adam(learning_rate=0.001, clipnorm=1.0)
    #TF: loss = tf.keras.losses.MeanAbsolutePercentageError()
    with open(f"normalization/{model_id}/z_scores.pkl", "rb") as ff:
        z_scores = pickle.load(ff)

    model = RouteNetGauss(
        output_dim=5,
        mask_field=f"flow_has_{metric}",
        inference_mode=inference_mode,
        use_trans_delay = metric == "delay",
        z_scores=z_scores,
    )
    #TF: model.compile(optimizer=optimizer, loss=loss)
    #TF: model.load_weights(f"ckpt/{model_id}/{checkpoint}").expect_partial()
    # PyTorch: the TF checkpoint was converted once by convert_tf_checkpoint.py (pure re-layout of
    # the 34 weight tensors); the z-score buffers are not part of a TF checkpoint and come from
    # z_scores.pkl above, hence strict=False.
    state = torch.load(f"ckpt/{model_id}/{checkpoint}.pt", map_location="cpu", weights_only=True)
    missing, unexpected = model.load_state_dict(state, strict=False)
    assert not unexpected and all(k.startswith("z_") for k in missing), (missing, unexpected)
    return model.to(DEVICE).eval()


In [ ]:
model_mawi_pcaps_delay = load_model(
    "paper_weights/mawi_pcaps/RouteNetGauss/delay",
    "201-19.7350",
    "delay",
)
model_mawi_pcaps_jitter = load_model(
    "paper_weights/mawi_pcaps/RouteNetGauss/jitter",
    "204-14.3108",
    "jitter",
)

model_trex_multiburst_delay = load_model(
    "paper_weights/trex_multiburst_filtered/RouteNetGauss/delay",
    "214-4.7122",
    "delay",
)
model_trex_multiburst_jitter = load_model(
    "paper_weights/trex_multiburst/RouteNetGauss/jitter",
    "242-11.5459",
    "jitter",
)

model_trex_synthetic_delay = load_model(
    "paper_weights/trex_synthetic_filtered/RouteNetGauss/delay",
    "221-2.8343",
    "delay",
)

model_trex_synthetic_jitter = load_model(
    "paper_weights/trex_synthetic/RouteNetGauss/jitter",
    "244-9.8195",
    "jitter",
)

## Evaluation results

In [ ]:
def mape(y_true, y_pred):
    return f"{np.mean(np.abs((y_true - y_pred) / y_true)) * 100:.3f}%"


def mae(y_true, y_pred):
    return f"{np.mean(np.abs((y_true - y_pred)) * 1e6):.3f}μs"


def r2(y_true, y_pred):
    return f"{1 - np.sum(np.square(y_true - y_pred)) / np.sum(np.square(y_true - np.mean(y_true))):.3f}"


#TF: def concatenate_ds(ds: tf.data.Dataset) -> np.ndarray:
def concatenate_ds(ds) -> np.ndarray:
    """Transforms dataset into a numpy array. Used for evaluation.

    Parameters
    ----------
    ds : utils.ListDataset (tf.data.Dataset in the TF version)
        Dataset to transform.

    Returns
    -------
    np.ndarray
        Numpy array with the concatenated targets.
    """
    res = [y.numpy() for _, y in iter(ds)]
    return np.concatenate(res, axis=0)


def predict(model: RouteNetGauss, ds) -> np.ndarray:
    """PyTorch replacement for `model.predict(ds)`: run the model on every scenario and
    concatenate the predictions (same order as `concatenate_ds`)."""
    with torch.no_grad():
        return np.concatenate([model(sample_to_device(x, DEVICE)).cpu().numpy() for x, _ in iter(ds)], axis=0)


def concatenate_ds_with_donor_mask(
    #TF: ds: tf.data.Dataset, mask_ds: tf.data.Dataset, metric: str = "jitter"
    ds, mask_ds: dict, metric: str = "jitter"
) -> np.ndarray:
    """Transforms dataset into a numpy array. Windows are selected to match mask_ds.
    Used for evaluation.

    Parameters
    ----------
    ds : utils.ListDataset (tf.data.Dataset in the TF version)
        Dataset to transform.

    mask_ds : dict
        ds_to_dict(...) of the dataset whose flow_has_<metric> masks select the windows.

    Returns
    -------
    np.ndarray
        Numpy array with the concatenated targets.
    """
    res = []
    targets = [
        f"flow_avg_{metric}",
        f"flow_p50_{metric}",
        f"flow_p90_{metric}",
        f"flow_p95_{metric}",
        f"flow_p99_{metric}",
    ]
    mask = f"flow_has_{metric}"
    for x, _ in iter(ds):
        #TF: mask_field = mask_ds[x["sample_idx"].numpy()][0][mask]
        mask_field = mask_ds[int(x["sample_idx"])][0][mask]
        #TF: reshaped_mask = tf.expand_dims(seg_to_global_reshape(mask_field, num_dims=2), 1)
        reshaped_mask = seg_to_global_reshape(mask_field, num_dims=2).unsqueeze(1)
        #TF: val = tf.concat(
        #TF:     [
        #TF:         tf.reshape(
        #TF:             tf.boolean_mask(seg_to_global_reshape(x[target]), reshaped_mask),
        #TF:             (-1, 1),
        #TF:         )
        #TF:         for target in targets
        #TF:     ],
        #TF:     axis=1,
        #TF: )
        val = torch.cat(
            [
                seg_to_global_reshape(x[target])[reshaped_mask].reshape(-1, 1)
                for target in targets
            ],
            dim=1,
        )
        res.append(val.numpy())
    total = np.concatenate(res, axis=0)
    total[total <= 0] = 0
    return total


def evaluate_model_vs_sim(
    #TF: true_ds: tf.data.Dataset,
    #TF: simulated_ds: tf.data.Dataset,
    true_ds,
    simulated_ds,
    model: RouteNetGauss,
    metric: str,
) -> None:
    """Generates a summary report comparing the model's predictions against the
    simulator's.

    Parameters
    ----------
    true_ds : utils.ListDataset (tf.data.Dataset in the TF version)
        Dataset representing the ground truth (testbed).
    simulated_ds : utils.ListDataset (tf.data.Dataset in the TF version)
        Dataset representing the simulator's prediction of the ground truth.
    model : RouteNetGauss
        Trained RouteNet-Gauss model.
    metric : str
        Name of perfomance metric [delay, jitter] to evaluate.
    """

    numpy_true_ds = concatenate_ds(true_ds)
    #TF: numpy_pred_ds = model.predict(true_ds)
    numpy_pred_ds = predict(model, true_ds)
    numpy_simulated_ds = concatenate_ds_with_donor_mask(
            simulated_ds, ds_to_dict(true_ds), metric.lower()
        )

    for ii, agg in enumerate(
        ["Average", "Median", "90th Percentile", "95th Percentile", "99th Percentile"]
    ):
        for err_metric in [mape, mae, r2]:
            print(
                f"{agg} {metric} ({err_metric.__name__}):",
                f"RouteNet-Gauss {err_metric(numpy_true_ds[:, ii], numpy_pred_ds[:, ii])}",
                f"OMNeT++ {err_metric(numpy_true_ds[:, ii], numpy_simulated_ds[:, ii])}",
            )
        print()


### TREX Synthetic

In [ ]:
evaluate_model_vs_sim(
    ds_trex_synthetic_delay,
    ds_trex_synthetic_simulated_delay,
    model_trex_synthetic_delay,
    "Delay",
)

In [ ]:
evaluate_model_vs_sim(
    ds_trex_synthetic_jitter,
    ds_trex_synthetic_simulated_jitter,
    model_trex_synthetic_jitter,
    "Jitter",
)

### TREX MULTIBURST

In [ ]:
evaluate_model_vs_sim(
    ds_trex_multiburst_delay,
    ds_trex_multiburst_simulated_delay,
    model_trex_multiburst_delay,
    "Delay",
)

In [ ]:
evaluate_model_vs_sim(
    ds_trex_multiburst_jitter,
    ds_trex_multiburst_simulated_jitter,
    model_trex_multiburst_jitter,
    "Jitter",
)

### MAWI PCAPS

In [ ]:
evaluate_model_vs_sim(
    ds_mawi_pcaps_delay,
    ds_mawi_pcaps_simulated_delay,
    model_mawi_pcaps_delay,
    "Delay",
)

In [ ]:
evaluate_model_vs_sim(
    ds_mawi_pcaps_jitter,
    ds_mawi_pcaps_simulated_jitter,
    model_mawi_pcaps_jitter,
    "Jitter",
)